## Setup

In [1]:
from pathlib import Path

In [3]:
import numpy as np
import pandas as pd
from Bio import SeqIO

In [4]:
from sklearn.preprocessing import LabelEncoder

In [6]:
data_dir = Path("../data")
raw_dir = data_dir / "raw"

for item in raw_dir.iterdir(): 
    print(item)

../data/raw/mc1r.txt
../data/raw/morfometricos.xlsx
../data/raw/cytb.txt
../data/raw/.gitkeep


In [8]:
mc1r_path = raw_dir / "mc1r.txt"
cytb_path = raw_dir / "cytb.txt"

assert mc1r_path.is_file() and cytb_path.is_file()

## Carga de datos

In [65]:
def fasta_to_df(filepath) -> pd.DataFrame:
    records = []

    for record in SeqIO.parse(filepath, "fasta"): 
        records.append({
            'id': record.id, 
            'sequence': str(record.seq), 
        })

    df = pd.DataFrame(records)
    return df

In [67]:
mc1r_df = fasta_to_df(mc1r_path)
mc1r_df.head()

,id,sequence
0,KR026455,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
1,KR026456,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
2,KR026457,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
3,KR026458,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
4,KR026459,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...


In [24]:
mc1r_df.shape

(43, 2)

In [25]:
cytb_df = fasta_to_df(cytb_path, 'mc1r')
cytb_df.head()

,id,sequence
0,KR026343,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...
1,KR026344,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...
2,KR026345,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...
3,KR026346,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...
4,KR026347,GCCTAATTATCCAAATCCTAACAGGACTATTTCTAGCCATACACTA...


In [26]:
cytb_df.shape

(36, 2)

- El indicador genético `mc1r` posee 43 registros, mientras que el indicador `cytb` cuenta con 36 registros. 

## Validación y limpieza

La longitud de las cadenas: 

In [27]:
mc1r_df['length'] = mc1r_df['sequence'].str.len()
cytb_df['length'] = cytb_df['sequence'].str.len()

In [29]:
mc1r_df['length'].value_counts()

length
815    43
Name: count, dtype: int64

- Todos los registros considerados para el indicador `mc1r` poseen cadenas con 815 nucleótidos

In [31]:
cytb_df['length'].value_counts()

length
1027    35
1081     1
Name: count, dtype: int64

- Uno de los 36 registros posee cadena con más nucleótidos que el resto. 

Verificamos el registro anormal: 

In [33]:
cytb_df.query("length != 1027")

,id,sequence,length
35,Holbrookia,CTTTGGTTCACTACTAGGAATTTGCCTGATTATTCAAATCTTAACA...,1081


Para evitar la introducción de ruido, eliminaremos este registro. 

In [37]:
cytb_df = cytb_df.drop(axis=0, index=35)

In [40]:
assert cytb_df['length'].nunique() == 1

Este registro también se encuentra en el dataset del indicador `mc1r`. Lo eliminaremos para ser congruentes. 

In [46]:
mc1r_df.iloc[-1]

id                                                 Holbrookia
sequence    CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...
length                                                    815
Name: 42, dtype: object

In [49]:
mc1r_df.query("id == 'Holbrookia'")

,id,sequence,length
42,Holbrookia,CGGAGTTGCAAATGACAAGGATGAGGTAGAGGCTGAAGTAGCTGAA...,815


In [51]:
mc1r_df = mc1r_df.drop(axis=0, index=42)

Los nucleótidos en cada registro deben ser el mismo: 

In [52]:
VALID_BASES = set('ACGT')

def validate_sequences(df):
    invalid = []

    for idx, seq in enumerate(df['sequence']):
        chars = set(seq)

        if not chars.issubset(VALID_BASES):
            invalid.append(idx)

    return invalid

In [57]:
assert not validate_sequences(mc1r_df), 'Nucleótido invalido'

In [56]:
assert not validate_sequences(cytb_df), 'Nucleótido invalido'

## Matriz genética

Para hacer inferencia, es necesario reprecentar los registros como una matriz genética. La idea es convertir las secuencias de nucleótidos en variables conparables.

In [60]:
def sequences_to_matrix(df):
    sequences = df['sequence'].tolist()

    matrix = pd.DataFrame(
        [list(seq) for seq in sequences]
    )

    matrix.columns = [i for i in range(matrix.shape[1])]

    matrix.insert(0, 'id', df['id'].values)

    return matrix

In [63]:
mc1r_matrix = sequences_to_matrix(mc1r_df)
mc1r_matrix.head()

,id,0,1,2,3,4,5,6,7,8,...,805,806,807,808,809,810,811,812,813,814
0,KR026455,C,G,G,A,G,T,T,G,C,...,G,T,C,A,C,T,G,G,C,A
1,KR026456,C,G,G,A,G,T,T,G,C,...,G,T,C,G,C,T,G,G,C,A
2,KR026457,C,G,G,A,G,T,T,G,C,...,G,T,C,G,C,T,G,G,C,A
3,KR026458,C,G,G,A,G,T,T,G,C,...,G,T,C,G,C,T,G,G,C,A
4,KR026459,C,G,G,A,G,T,T,G,C,...,G,T,C,G,C,T,G,G,C,A


In [64]:
cytb_matrix = sequences_to_matrix(cytb_df)
cytb_matrix.head()

,id,0,1,2,3,4,5,6,7,8,...,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026
0,KR026343,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T
1,KR026344,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T
2,KR026345,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T
3,KR026346,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T
4,KR026347,G,C,C,T,A,A,T,T,A,...,G,A,T,A,A,A,A,T,T,T


## Codofificación numérica

Para trabajar con modelos matemáticos, es necesario reprecentar la información genética de manera numérica. Haremos esto mapeando el identificador de nucleótido a un número entero: 

    > {A: 1, C: 1, G: 2, T: 3}

In [76]:
ENCODING = {
    'A': 0,
    'C': 1,
    'G': 2,
    'T': 3
}

def encode_matrix(df):
    encoded = df.copy()

    feature_cols = encoded.columns[1:]

    for col in feature_cols:
        encoded[col] = encoded[col].map(ENCODING)

    return encoded

In [77]:
mc1r_encoded = encode_matrix(mc1r_matrix)
mc1r_encoded.head()

,id,0,1,2,3,4,5,6,7,8,...,805,806,807,808,809,810,811,812,813,814
0,KR026455,1,2,2,0,2,3,3,2,1,...,2,3,1,0,1,3,2,2,1,0
1,KR026456,1,2,2,0,2,3,3,2,1,...,2,3,1,2,1,3,2,2,1,0
2,KR026457,1,2,2,0,2,3,3,2,1,...,2,3,1,2,1,3,2,2,1,0
3,KR026458,1,2,2,0,2,3,3,2,1,...,2,3,1,2,1,3,2,2,1,0
4,KR026459,1,2,2,0,2,3,3,2,1,...,2,3,1,2,1,3,2,2,1,0


In [78]:
cytb_encoded = encode_matrix(cytb_matrix)
cytb_encoded.head()

,id,0,1,2,3,4,5,6,7,8,...,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026
0,KR026343,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3
1,KR026344,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3
2,KR026345,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3
3,KR026346,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3
4,KR026347,2,1,1,3,0,0,3,3,0,...,2,0,3,0,0,0,0,3,3,3


## Exportación

Guardamos ambos datasets en el directorio `data/processed`: 

In [80]:
cytb_output_path = Path("../data/processed/cytb_encoded.csv")
cytb_encoded.to_csv(
    cytb_output_path,
    index=False
)

In [81]:
mc1r_output_path = Path("../data/processed/mc1r_encoded.csv")
mc1r_encoded.to_csv(
    mc1r_output_path,
    index=False
)